In [ ]:
from_config = True
df = None # Dummy value
id_image = 0
all_data = False
batch_size = 32
val_split = 0.2
generate_one_element_pt = False
generate_batches_pt = False

### Digit Recognizer

An implementation using JAX library of the DataSet and the CNN Models, assumes the dataset in question is a dataframe with flattened images. 

#### 1) The Dataset 

In [ ]:
from mlops.datamodel.data_loader import DM_Loader 

dml = DM_Loader()

"""
Constructing the Data Model Dataset
"""
# If you want to set data from config : 
# ------------
# with open("utils/project_headers.json", "r") as f :
#     PROJECT_HEADERS = json.load(f)
# ------------
if from_config:
    dml.set_new_data_from_config() 

# If you want to set it yourself 
# dml.set_new_data(df : pd.DataFrame)
else :
    dml.set_new_data(df)

"""
Getting the data
"""
# Either you want to get the entire dataframe
if all_data : 
    __df = dml.get_full_data()
# or just one element from the data : formatted as an image (getter from the id in the dataframe) 
else :
    __image, label = dml.get_data_at_id(id_image) # Label will be none in case data is a test data

"""
Getting the generators
"""

# Either you want to loop through all elements, once each : 
if generate_one_element_pt : 
    generator = dml.generate_data_as_element()
# Or you want to get batches 
elif generate_batches_pt : 
    generator = dml.generate_data_as_batch(batch_size)
# Or you want to get a train val split (with batch_size)
else : 
    train_generator, val_generator , train_size, val_size = dml.generate_data_as_train_test_split(batch_size, val_split)

#### 2) The models 

In [ ]:
# For now the only model is LeNet. More very soon. 

batch = next(train_generator)

from mlops.model.convolutional.LeNet import LeNet

ln = LeNet()

## To construct the model (basically so that it understands the input shape) , you can use the .construct method 
ln.construct(batch[0]) ##. Only the training data without labels

## The model is constructed, you can now try to predict something (not trained)

probabilities, predictions = ln.predict(batch[0])
predictions

In [ ]:
# You can now also train the model
ln.train(dml,epochs = 1, batch_size= 32) # You give it the entire dataset and it will manage by default the train test split generators 

In [ ]:
# If the model is trained, you can also save it
ln.save("your_dir/vlenet.1.0.pkl")

In [ ]:
# You can also load an existing model (NOTE : Has to be the same structure for now)
ln2 = LeNet()
ln2.load("your_dir/vlenet.1.0.pkl")

As an important thing : It is better that the root dir where you store the models have __ before the name so that it is automatically ignored by git